# P2 — Analyse exploratoire : Mobilité urbaine (CityBikes API v2 + Historique 5 ans)

**Responsable : P2**

Ce notebook exploite directement l'entrepôt ClickHouse (`urbanhub.velo_releve` et les vues analytiques `v_velo_*`, sans limite de volume sur 5 ans) — avec repli automatique sur les fichiers Parquet locaux si ClickHouse est éteint — pour répondre aux **5 questions métier** :
1. Quelles stations présentent le plus fort taux d’utilisation ?
2. Peut-on identifier des zones où l’offre de vélos est insuffisante ?
3. Quels sont les pics d’utilisation journaliers du réseau de vélos ?
4. Existe-t-il des déséquilibres géographiques dans la disponibilité des vélos ?
5. Peut-on détecter des stations critiques nécessitant un rééquilibrage des vélos ?

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import plotly.express as px

from common.config import get_settings, load_ref_ville
from common.warehouse import get_warehouse
from p2_velos.analytics import (
    daily_usage_peaks,
    detect_critical_stations,
    detect_undersupplied_zones,
    ensure_enriched_dataframe,
    geographic_imbalance,
    top_utilisation_stations,
)

ref_villes = load_ref_ville()
city_centers = {v['ville_id']: (v['lat'], v['lon']) for v in ref_villes}
wh = get_warehouse()
USE_CLICKHOUSE = wh.ping()
print(f'Villes configurées ({len(ref_villes)}) | ClickHouse connecté : {USE_CLICKHOUSE}')

Villes configurées (12) | ClickHouse connecté : True


## 0. Contrôle de la couverture temporelle et volumétrie (fenêtre 5 ans)

> Si seule la journée en cours apparaît, chargez l'historique officiel `data.citybik.es` dans ClickHouse via :
> `python -m p2_velos.backfill --years 5` (ou `--max-months-per-city 1` pour un premier mois rapide sur les 12 villes).

In [2]:
df_local = pd.DataFrame()

if USE_CLICKHOUSE:
    meta = wh.query_df('''
        SELECT
            count() AS total_releves,
            uniqExact(ville_id) AS nb_villes,
            uniqExact(station_id) AS nb_stations,
            min(timestamp) AS debut_utc,
            max(timestamp) AS fin_utc,
            uniqExact(toYYYYMM(timestamp)) AS nb_mois
        FROM urbanhub.velo_releve
        WHERE timestamp >= now() - INTERVAL 5 YEAR
    ''')
    display(meta)
    summary_city = wh.query_df('''
        SELECT
            ville_id,
            uniqExact(station_id) AS nb_stations,
            count() AS nb_releves,
            min(timestamp) AS debut_utc,
            max(timestamp) AS fin_utc,
            round(avg(bikes_available), 1) AS velos_moyens,
            round(avg(taux_utilisation) * 100, 1) AS taux_utilisation_pct,
            round(countIf(est_critique = 1) / count() * 100, 1) AS pct_critiques
        FROM urbanhub.velo_releve
        WHERE timestamp >= now() - INTERVAL 5 YEAR
        GROUP BY ville_id
        ORDER BY nb_releves DESC
    ''')
    display(summary_city)
else:
    base_silver = Path(get_settings().local_data_dir) / 'silver'
    frames = []
    for sub in ('velos', 'velos_historique'):
        d = base_silver / sub
        if d.exists():
            for p in sorted(d.rglob('velo_releve_*.parquet')):
                frames.append(pd.read_parquet(p))
            for p in sorted(d.rglob('velo_releve_*.jsonl')):
                frames.append(pd.read_json(p, lines=True))
    df_local = ensure_enriched_dataframe(pd.concat(frames, ignore_index=True)) if frames else pd.DataFrame()
    print(f'Mode local : {len(df_local):,} relevés ({df_local["timestamp"].min()} -> {df_local["timestamp"].max()})')

,total_releves,nb_villes,nb_stations,debut_utc,fin_utc,nb_mois
0,0,0,0,1970-01-01,1970-01-01,0


""


### Échantillon des 7 variables obligatoires dans ClickHouse (`velo_releve`)

In [4]:
required_cols = ['station_id', 'station_name', 'latitude', 'longitude', 'bikes_available', 'free_slots', 'timestamp']
if USE_CLICKHOUSE:
    sample_df = wh.query_df(
        'SELECT ville_id, station_id, station_name, latitude, longitude, bikes_available, free_slots, timestamp '
        'FROM urbanhub.velo_releve ORDER BY timestamp DESC LIMIT 10'
    )
else:
    sample_df = df_local[['ville_id'] + required_cols].head(10)
expected_cols = ['ville_id'] + required_cols

# ClickHouse may return an empty DataFrame without schema when no rows match.
if sample_df.empty and len(sample_df.columns) == 0:
    sample_df = pd.DataFrame(columns=expected_cols)

assert all(c in sample_df.columns for c in required_cols)
sample_df

,ville_id,station_id,station_name,latitude,longitude,bikes_available,free_slots,timestamp


## Question 1 : Quelles stations présentent le plus fort taux d’utilisation ?

In [5]:
if USE_CLICKHOUSE:
    top_stations = wh.query_df('''
        SELECT
            ville_id,
            station_name,
            capacite_moyenne,
            round(taux_utilisation_moyen * 100, 1) AS taux_utilisation_pct,
            volatilite_velos,
            amplitude_velos,
            nb_releves
        FROM urbanhub.v_velo_utilisation_stations
        ORDER BY taux_utilisation_moyen DESC, volatilite_velos DESC, capacite_moyenne DESC
        LIMIT 20
    ''')
else:
    top_stations = top_utilisation_stations(df_local, top_n=20)

fig1 = px.bar(
    top_stations.sort_values('taux_utilisation_pct'),
    x='taux_utilisation_pct',
    y='station_name',
    color='ville_id',
    orientation='h',
    hover_data=['capacite_moyenne', 'volatilite_velos', 'amplitude_velos', 'nb_releves'],
    title="Top 20 des stations présentant le plus fort taux d'utilisation",
    labels={'taux_utilisation_pct': "Taux d'utilisation moyen (%)", 'station_name': 'Station'},
    height=600,
)
fig1.show()
top_stations

KeyError: 'taux_utilisation_pct'

## Question 2 : Peut-on identifier des zones où l’offre de vélos est insuffisante ?

In [ ]:
if USE_CLICKHOUSE:
    zones_sous_offre = wh.query_df('''
        SELECT
            ville_id,
            round(latitude, 2) AS zone_lat,
            round(longitude, 2) AS zone_lon,
            uniqExact(station_id) AS nb_stations,
            any(station_name) AS exemple_station,
            round(avg(bikes_available), 2) AS velos_dispo_moyens,
            round(countIf(est_vide = 1) / count() * 100, 1) AS pct_temps_vide,
            round(countIf(taux_occupation <= 0.15) / count() * 100, 1) AS pct_temps_sous_offre,
            count() AS nb_releves
        FROM urbanhub.velo_releve
        GROUP BY ville_id, zone_lat, zone_lon
        ORDER BY pct_temps_sous_offre DESC, pct_temps_vide DESC, nb_stations DESC
        LIMIT 100
    ''')
else:
    zones_sous_offre = detect_undersupplied_zones(df_local, grid_precision=2, low_supply_threshold=0.15)

fig2 = px.scatter_map(
    zones_sous_offre.head(100),
    lat='zone_lat',
    lon='zone_lon',
    size='nb_stations',
    color='pct_temps_sous_offre',
    color_continuous_scale='YlOrRd',
    hover_name='exemple_station',
    hover_data=['ville_id', 'nb_stations', 'velos_dispo_moyens', 'pct_temps_vide'],
    map_style='carto-positron',
    zoom=5,
    height=520,
    title='Zones à offre de vélos insuffisante (<= 15 % de vélos disponibles)',
)
fig2.show()
zones_sous_offre.head(15)

## Question 3 : Quels sont les pics d’utilisation journaliers du réseau (et évolution mensuelle) ?

In [ ]:
if USE_CLICKHOUSE:
    peaks = wh.query_df('''
        SELECT
            ville_id,
            heure_locale,
            if(type_jour = 'weekend', 'Week-end', 'Semaine') AS type_jour,
            round(taux_utilisation_moyen * 100, 1) AS taux_utilisation_pct,
            round(pct_stations_vides * 100, 1) AS pct_stations_vides,
            round(pct_stations_pleines * 100, 1) AS pct_stations_pleines,
            nb_releves
        FROM urbanhub.v_velo_pics_horaires
        ORDER BY ville_id, type_jour, heure_locale
    ''')
    monthly = wh.query_df('''
        SELECT
            ville_id,
            formatDateTime(mois_utc, '%Y-%m') AS mois,
            round(taux_utilisation_moyen * 100, 1) AS taux_utilisation_pct,
            round(pct_stations_critiques * 100, 1) AS pct_critiques,
            nb_releves
        FROM urbanhub.v_velo_tendance_mensuelle
        ORDER BY mois, ville_id
    ''')
else:
    peaks = daily_usage_peaks(df_local, tz='Europe/Paris')
    monthly = pd.DataFrame()

fig3 = px.line(
    peaks,
    x='heure_locale',
    y='taux_utilisation_pct',
    color='ville_id',
    line_dash='type_jour',
    markers=True,
    title="Pics journaliers d'utilisation (heure locale Europe/Paris)",
    labels={'heure_locale': 'Heure (0-23h)', 'taux_utilisation_pct': "Taux d'utilisation (%)"},
)
fig3.update_xaxes(dtick=1, range=[0, 23])
fig3.show()

if not monthly.empty and monthly['mois'].nunique() > 1:
    fig3b = px.line(
        monthly,
        x='mois',
        y='taux_utilisation_pct',
        color='ville_id',
        markers=True,
        title="Évolution mensuelle du taux d'utilisation (historique ClickHouse)",
    )
    fig3b.show()

peaks.sort_values('taux_utilisation_pct', ascending=False).head(15)

## Question 4 : Existe-t-il des déséquilibres géographiques dans la disponibilité des vélos ?

In [ ]:
if USE_CLICKHOUSE:
    desequilibre = wh.query_df('''
        SELECT
            ville_id,
            secteur_geo,
            nb_stations,
            round(taux_occupation_moyen * 100, 1) AS taux_occupation_pct,
            round(pct_temps_vide * 100, 1) AS pct_temps_vide,
            round(pct_temps_pleine * 100, 1) AS pct_temps_pleine,
            distance_centre_km
        FROM urbanhub.v_velo_desequilibre_geo
    ''')
    city_avg = desequilibre.groupby('ville_id')['taux_occupation_pct'].transform('mean')
    desequilibre['ecart_moyenne_ville_pts'] = (desequilibre['taux_occupation_pct'] - city_avg).round(1)
    desequilibre = desequilibre.sort_values(['ville_id', 'ecart_moyenne_ville_pts'])
else:
    desequilibre = geographic_imbalance(df_local, city_centers=city_centers, center_radius_km=1.5)

fig4 = px.bar(
    desequilibre,
    x='secteur_geo',
    y='ecart_moyenne_ville_pts',
    color='ville_id',
    barmode='group',
    title='Écart de disponibilité par secteur géographique (Centre vs quadrants périphériques)',
    labels={'secteur_geo': 'Secteur', 'ecart_moyenne_ville_pts': 'Écart vs moyenne ville (pts %)'},
)
fig4.add_hline(y=0, line_dash='dash', line_color='gray')
fig4.show()
desequilibre

## Question 5 : Détection des stations critiques nécessitant un rééquilibrage

In [ ]:
if USE_CLICKHOUSE:
    critiques = wh.query_df('''
        SELECT
            ville_id,
            station_id,
            station_name,
            latitude,
            longitude,
            dernier_timestamp AS timestamp,
            bikes_available,
            free_slots,
            capacity,
            round(dernier_taux_occupation * 100, 1) AS taux_occupation_pct,
            action_reequilibrage,
            delta_velos_equilibre,
            round(pct_temps_critique * 100, 1) AS pct_temps_critique,
            round(abs(dernier_taux_occupation - 0.5) * 100 + pct_temps_critique * 35 + least(capacity, 50) * 0.3, 1) AS score_priorite
        FROM urbanhub.v_velo_stations_critiques
        WHERE action_reequilibrage != 'AUCUNE'
        ORDER BY score_priorite DESC
    ''')
else:
    critiques = detect_critical_stations(df_local, low_threshold=0.10, high_threshold=0.90)

print('Répartition des actions recommandées :')
print(critiques['action_reequilibrage'].value_counts())

fig5 = px.scatter_map(
    critiques,
    lat='latitude',
    lon='longitude',
    color='action_reequilibrage',
    size='capacity',
    color_discrete_map={'REAPPROVISIONNER': '#d62728', 'DESENGORGER': '#1f77b4'},
    hover_name='station_name',
    hover_data=['ville_id', 'bikes_available', 'free_slots', 'delta_velos_equilibre', 'score_priorite'],
    map_style='carto-positron',
    zoom=5,
    height=520,
    title='Stations critiques nécessitant une intervention de rééquilibrage',
)
fig5.show()
critiques.head(20)

## Synthèse & recommandations opérationnelles

- **Planification du rééquilibrage** : prioriser les stations à fort `score_priorite` en déplaçant les vélos des stations à `DESENGORGER` (delta négatif) vers les stations proches à `REAPPROVISIONNER` (delta positif).
- **Dimensionnement de l'offre** : les zones identifiées en Question 2 présentant un taux de sous-offre persistant (> 50 % du temps sous 15 % de vélos) sont candidates à une extension de capacité ou à l'ajout de stations.
- **Anticipation des pics** : programmer les tournées de régulation 30 à 45 minutes avant les pics pendulaires détectés en Question 3 (généralement 8h-9h et 17h30-19h en semaine).
- **Couverture historique** : l'archive officielle CityBikes (`data.citybik.es`, ODbL v1.0) débute en novembre 2024 et est complétée en continu par le flux streaming toutes les 2-3 minutes (`api.citybik.es/v2`).